In [18]:
import os
import numpy as np
import pandas as pd
# from constants import *

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt

from warnings import filterwarnings
filterwarnings("ignore")

pd.set_option("display.max_rows",100)
pd.set_option("display.max_columns",100)

In [35]:
folder_path = r"D:\Attrition\Data\Raw"
file_name = r"sample_raw.xlsx"

df = pd.read_excel(os.path.join(folder_path, file_name))
print(df.shape)
df.head()

(127, 9)


,Customernumber,LastDepositMonth,ReferenceDate,Attrition_flag,University,Gender,CustomerAge@PostDate,AgeWithWeyay,FullKYC_Y_N
0,112002943,2022-12-31 21:00:00,2022-12-31 21:00:00,0,Private,F,18.7,13,1
1,112002943,2023-01-31 21:00:00,2023-01-31 21:00:00,0,Private,F,18.8,14,1
2,112002943,2023-02-28 21:00:00,2023-02-28 21:00:00,0,Private,F,18.9,15,1
3,112002943,2023-03-31 21:00:00,2023-03-31 21:00:00,0,Private,F,19.0,16,1
4,112002943,2023-04-30 21:00:00,2023-04-30 21:00:00,0,Private,F,19.0,17,1


In [36]:
# dropping unwanted columns
# --> drop Allowance month
df.drop(columns = ["AllowanceMonth", "ReferenceDate"], inplace=True)

# ReferenceDate --> MonthKey
df['ReferenceDate'] = pd.to_datetime(df['MonthKey'])
df["ReferenceMonth"] = df['ReferenceDate'].dt.month
df["ReferenceQuarter"] = df['ReferenceDate'].dt.quarter
df["ReferenceYear"]= df['ReferenceDate'].dt.year
df['LastDepositMonth'] = pd.to_datetime(df['LastDepositMonth'].dt.date)
df['ReferenceDate'] = pd.to_datetime(df['ReferenceDate'].dt.date)

df.drop(columns = ["MonthKey"], inplace=True)

df.head()

,Customernumber,LastDepositMonth,ReferenceDate,Attrition_flag,University,Gender,CustomerAge@PostDate,AgeWithWeyay,FullKYC_Y_N,ReferenceMonth,ReferenceQuarter,ReferenceYear
0,112002943,2022-12-31,2022-12-31,0,Private,F,18.7,13,1,12,4,2022
1,112002943,2023-01-31,2023-01-31,0,Private,F,18.8,14,1,1,1,2023
2,112002943,2023-02-28,2023-02-28,0,Private,F,18.9,15,1,2,1,2023
3,112002943,2023-03-31,2023-03-31,0,Private,F,19.0,16,1,3,1,2023
4,112002943,2023-04-30,2023-04-30,0,Private,F,19.0,17,1,4,2,2023


In [39]:
def fill_null_with_median(dataframe, cols):
    for col_ in cols:
        if dataframe[col_].isnull().sum() > 0:
            dataframe[col_] = dataframe[col_].fillna(dataframe[col_].median())
    return dataframe

def fill_null_with_ffill(dataframe, cols):
    df['A'] = df['A'].ffill()
    for col_ in cols:
        dataframe[col_] = dataframe[col_].ffill()
    return dataframe


def fill_null_with_zero(dataframe, cols):
    for col_ in cols:
        if dataframe[col_].isnull().sum() > 0:
            dataframe[col_] = dataframe[col_].fillna(0)
    return dataframe

In [ ]:
# all customer number should have same max_date --> ReferenceDate

In [31]:
def prep_data(df):
    model_data = pd.DataFrame()
    Train_agg_df = pd.DataFrame()
    train_agg_df, test_agg_df = pd.DataFrame(), pd.DataFrame()
    
    for cust_id, df_slice in df.groupby('Customernumber'):
    
        print(f"Running for Cutomer {cust_id}")

        min_allowance_date = df_slice["LastDepositMonth"].min()
        df_slice = df_slice[df_slice["ReferenceDate"]>=min_allowance_date]


        # ffill  columns --> LastDepositMonth , Gender, Univ,  Flag 
        df_slice = fill_null_with_ffill(df_slice, ["LastDepositMonth", "Gender", "University", "Attrition_flag", "CustomerAge@PostDate"])

        # Flag Active-0, Attrition-1
        df_slice["Attrition_flag"] = np.where(df_slice["Flag"]=="Active", 0, 1)

        # any spend which is null --> make it 0
        df_slice = fill_null_with_zero(df_slice, ["PrePaidSpends" , "DebitSpends"])
        
        # CustomerAge@PostDate increment with 0.1
        
        min_date, max_date = df_slice["ReferenceDate"].min(),df_slice["ReferenceDate"].max()
    
        diff = relativedelta(max_date, min_date)
        months_diff = diff.years * 12 + diff.months
    
        # assert months_diff + 1 == df_slice.shape[0]
    
        df_slice = fill_null(df_slice)
        df_slice = df_slice.sort_values(by="ReferenceDate")
    
        # Allowance Deposit
        df_slice['months_since_last_deposit'] = (df_slice['ReferenceDate'].dt.year - df_slice['LastDepositMonth'].dt.year) * 12 + \
                                                (df_slice['ReferenceDate'].dt.month - df_slice['LastDepositMonth'].dt.month)
    
        # Balance 
        df_slice['last_3m_avg_balance'] = df_slice['BALANCE_COL'].rolling(window=3, min_periods=1).mean()
    
        df_slice["yearly_balance"] = df_slice.groupby("ReferenceYear")["BALANCE_COL"].transform(sum)
        df_slice["quarterly_balance"] = df_slice.groupby(["ReferenceYear","ReferenceQuarter"])["BALANCE_COL"].transform(sum)
    
        df_slice["balance_monthly_index"] = df_slice["BALANCE_COL"]/df_slice["yearly_balance"]
        df_slice["balance_quarterly_index"] = df_slice["BALANCE_COL"]/df_slice["quarterly_balance"]
    
        # Debit Spend 
        df_slice['last_3m_avg_d_spend'] = df_slice['DEBIT_SPEND_COL'].rolling(window=3, min_periods=1).mean()
    
        df_slice["yearly_d_spend"] = df_slice.groupby("ReferenceYear")["DEBIT_SPEND_COL"].transform(sum)
        df_slice["quarterly_d_spend"] = df_slice.groupby(["ReferenceYear","ReferenceQuarter"])["DEBIT_SPEND_COL"].transform(sum)
    
        df_slice["d_spend_monthly_index"] = df_slice["DEBIT_SPEND_COL"]/df_slice["yearly_d_spend"]
        df_slice["d_spend_quarterly_index"] = df_slice["DEBIT_SPEND_COL"]/df_slice["quarterly_d_spend"]
    
        # PP Spend 
        df_slice['last_3m_avg_p_spend'] = df_slice['P_SPEND_COL'].rolling(window=3, min_periods=1).mean()
    
        df_slice["yearly_p_spend"] = df_slice.groupby("ReferenceYear")["P_SPEND_COL"].transform(sum)
        df_slice["quarterly_p_spend"] = df_slice.groupby(["ReferenceYear","ReferenceQuarter"])["P_SPEND_COL"].transform(sum)
    
        df_slice["p_spend_monthly_index"] = df_slice["P_SPEND_COL"]/df_slice["yearly_p_spend"]
        df_slice["p_spend_quarterly_index"] = df_slice["P_SPEND_COL"]/df_slice["quarterly_p_spend"]
        
        # Login
        df_slice["yearly_logins"] = df_slice.groupby("ReferenceYear")["LOGIN_COUNT_COL"].transform(sum)
        df_slice["quarterly_logins"] = df_slice.groupby(["ReferenceYear","ReferenceQuarter"])["LOGIN_COUNT_COL"].transform(sum)
    
        df_slice["logins_monthly_index"] = df_slice["LOGIN_COUNT_COL"]/df_slice["yearly_logins"]
        df_slice["logins_quarterly_index"] = df_slice["LOGIN_COUNT_COL"]/df_slice["quarterly_logins"]

        # Shifting Attrition flag for target variable
        df_slice["target"] = list(df_slice["Attrition_flag"])[2:] + [np.nan, np.nan]

        # target available dataframe
        assert df_slice[df_slice["target"].isna()].shape[0] == 2
        Train_df = df_slice[~(df_slice["target"].isna())]
        train_df = Train_df[~(Train_df["ReferenceDate"]> (train_df["ReferenceDate"].max() - relativedelta(months=2)))]
        test_df = Train_df[Train_df["ReferenceDate"]> (train_df["ReferenceDate"].max() - relativedelta(months=2))]
        
        df_model = pd.concat([df_model, df_slice], axis=1)
        Train_agg_df = pd.concat([Train_agg_df, Train_df], axis=1)
        train_agg_df = pd.concat([train_agg_df, train_df], axis=1)
        test_agg_df = pd.concat([test_agg_df, test_df], axis=1)
        
        break
        
    return df_model, Train_agg_df, train_agg_df, test_agg_df

Running for Cutomer 112002943
